# Lab 01 - Explore Azure Databricks - Toronto Transit Commission

This notebook is a custom adaptation of the Microsoft Learn DP-750T00 lab for Azure Databricks. Instead of using the sample dataset from the course, this version uses a real dataset from the [City of Toronto Open Data](https://open.toronto.ca/) catalog, with a focus on Toronto Transit Commission data.

The goal of this lab is to explore both the Azure Databricks workspace experience and the core steps involved in working with external data in Unity Catalog.

**In this lab, you will:**
* Create a catalog, schema, volume, and table
* Ingest a real open dataset into a Unity Catalog volume
* Copy into a table from files stored in a volume
* Practice basic Python exercises in Databricks notebooks
* Practice basic SQL exercises in Databricks notebooks

**Source dataset:** [TTC Routes and Schedules](https://open.toronto.ca/dataset/ttc-routes-and-schedules/) — City of Toronto Open Data Portal, published by the TTC, refreshed monthly, licensed under the [Open Government Licence – Toronto](https://open.toronto.ca/open-data-licence/).


##  Create a catalog, schema, volume and table

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS adb_dp750;

CREATE SCHEMA IF NOT EXISTS adb_dp750.default;

CREATE VOLUME IF NOT EXISTS adb_dp750.default.lab_data;

CREATE TABLE IF NOT EXISTS adb_dp750.default.ttc_routes;
  

## Ingest a real open dataset into a Unity Catalog volume

The dataset is published as a GTFS `.zip` file (routes, stops, trips, stop times, calendar). For this lab, the `routes.txt`, data is the focus so download the zip, extract that one file, and load it as a delta table.

In [0]:
%python
import requests
import zipfile
import io

catalog_name = "adb_dp750"
schema_name = "default"
volume_name = "lab_data"
volume_path = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}"

# TTC Routes and Schedules (GTFS zip) -- City of Toronto Open Data Portal
gtfs_url = (
    "https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/"
    "7795b45e-e65a-4465-81fc-c36b9dfff169/resource/"
    "cfb6b2b8-6191-41e3-bda1-b175c51148cb/download/opendata_ttc_schedules.zip"
)

response = requests.get(gtfs_url)
response.raise_for_status()

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    print("Files in the GTFS feed:", z.namelist())
    with z.open("routes.txt") as f:
        routes_csv_bytes = f.read()

# Save routes.txt to the Unity Catalog volume so it persists like the original lab's routes.csv
dbutils.fs.put(f"{volume_path}/ttc_routes.csv", routes_csv_bytes.decode("utf-8"), overwrite=True)
print(f"Saved to {volume_path}/ttc_routes.csv")

## Copy into a table from files stored in a volume

For existing Delta table, the syntax **COPY INTO** is the usual Databricks pattern. This is used to load files from storafe into an existing Delta table.

In [0]:
%sql
COPY INTO adb_dp750.default.ttc_routes
FROM '/Volumes/adb_dp750/default/lab_data/ttc_routes.csv'
FILEFORMAT = CSV
FORMAT_OPTIONS ("header" = "true")
COPY_OPTIONS ('mergeSchema' = 'true')

## Practice basic Python exercises in Databricks notebooks

**Task:** Pick a real TTC route from the ingested data, store its details in variables, and print a formatted summary — same pattern as the original lab, but with real data instead of made-up values.

#### Python 1: Load the table into a DataFrame, print the schema, row count, and 5 sample rows

In [0]:


df = spark.read.table("adb_dp750.default.ttc_routes")
display(df.printSchema())

print(df.count())

display(df.limit(5))


#### Python 2: Filter routes where route_type matches one mode, then count how many routes are in that mode

In [0]:
df = spark.read.table("adb_dp750.default.ttc_routes")
df2 = df.filter(df.route_type==0)
print(df2.count())

#### Python 3: Create a small lookup mapping GTFS route_type codes to readable labels, then add a derived label column and display the result

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("adb_dp750.default.ttc_routes")

route_type_lookup = F.create_map(
    F.lit(0), F.lit("Tram/Streetcar"),
    F.lit(1), F.lit("Subway/Metro"),
    F.lit(3), F.lit("Bus")
)

routes_with_labels_df = df.withColumn(
    "route_type_label",
    F.coalesce(route_type_lookup[F.col("route_type")], F.lit("Unknown"))
)

display(
    routes_with_labels_df.select(
        "route_short_name",
        "route_long_name",
        "route_type",
        "route_type_label"
    ).limit(10)
)


## Practice basic SQL exercises in Databricks notebooks

#### SQL 1: Return the first 10 routes with route_short_name, route_long_name, and route_type

In [0]:
%sql
SELECT route_short_name, route_long_name, route_type
FROM adb_dp750.default.ttc_routes
LIMIT 10


#### SQL 2: Group by route_type and count routes by type, ordered from highest to lowest

In [0]:
%sql
SELECT route_type, count(*) as cnt
FROM adb_dp750.default.ttc_routes
GROUP BY route_type
ORDER BY cnt DESC

#### SQL 3: Find routes with missing or blank route_desc, route_color, or route_text_color to practice basic data quality checks

In [0]:
%sql
SELECT *
FROM adb_dp750.default.ttc_routes
WHERE route_desc IS NULL 
OR route_color IS NULL
OR route_color IS NULL 
OR route_text_color IS NULL

# **End of the lab exercise. See you on the next one!**